In [21]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Load the Titanic dataset
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

# Preprocess the data
def preprocess_data(data, is_train=True):
    data['Title'] = data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    data['Title'] = data['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    data['Title'] = data['Title'].replace('Mlle', 'Miss')
    data['Title'] = data['Title'].replace('Ms', 'Miss')
    data['Title'] = data['Title'].replace('Mme', 'Mrs')
    
    data['Embarked'] = data['Embarked'].fillna('S')
    data['Fare'] = data['Fare'].fillna(data['Fare'].median())
    
    age_avg = data['Age'].mean()
    age_std = data['Age'].std()
    age_null_count = data['Age'].isnull().sum()
    age_null_random_list = np.random.randint(age_avg - age_std, age_avg + age_std, size=age_null_count)
    data.loc[data['Age'].isnull(), 'Age'] = age_null_random_list
    data['Age'] = data['Age'].astype(int)
    
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = 1
    data.loc[data['FamilySize'] > 1, 'IsAlone'] = 0
    
    data['Age*Class'] = data['Age'] * data['Pclass']
    data['Fare_per_Person'] = data['Fare'] / data['FamilySize']
    
    features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'Age*Class', 'Fare_per_Person']
    if is_train:
        features.append('Survived')
    
    data = data[features]
    
    data = pd.get_dummies(data)
    
    return data

train_data = preprocess_data(train_data, is_train=True)
test_data = preprocess_data(test_data, is_train=False)

# Scale the numerical features
scaler = StandardScaler()
train_data[['Age', 'Fare', 'FamilySize', 'Age*Class', 'Fare_per_Person']] = scaler.fit_transform(train_data[['Age', 'Fare', 'FamilySize', 'Age*Class', 'Fare_per_Person']])
test_data[['Age', 'Fare', 'FamilySize', 'Age*Class', 'Fare_per_Person']] = scaler.transform(test_data[['Age', 'Fare', 'FamilySize', 'Age*Class', 'Fare_per_Person']])

# Split the data into features and target
X_train = train_data.drop('Survived', axis=1)
y_train = train_data['Survived']
X_test = test_data.copy()

# Create an ensemble of classifiers
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
gb_clf = GradientBoostingClassifier(n_estimators=100, random_state=42)
voting_clf = VotingClassifier(estimators=[('rf', rf_clf), ('gb', gb_clf)], voting='soft')

# Perform hyperparameter tuning using GridSearchCV
param_grid = {
    'rf__max_depth': [4, 5, 6],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4],
    'gb__max_depth': [3, 4, 5],
    'gb__learning_rate': [0.1, 0.05, 0.01],
    'gb__subsample': [0.8, 0.9, 1.0]
}
grid_search = GridSearchCV(estimator=voting_clf, param_grid=param_grid, cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

# Train the model with the best hyperparameters
best_clf = grid_search.best_estimator_
best_clf.fit(X_train, y_train)

# Make predictions on the test data
predictions = best_clf.predict(X_test)



# Evaluate the model using cross-validation
cv_scores = cross_val_score(best_clf, X_train, y_train, cv=10)
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean cross-validation score: {cv_scores.mean():.4f}")

In [20]:
# Save the predictions to a CSV file    
test_data = pd.read_csv('test.csv')
output = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Survived': predictions})



In [15]:
output

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Load the Titanic dataset
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

# Preprocess the data
def preprocess_data(data):
    data['Title'] = data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    data['Title'] = data['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    data['Title'] = data['Title'].replace('Mlle', 'Miss')
    data['Title'] = data['Title'].replace('Ms', 'Miss')
    data['Title'] = data['Title'].replace('Mme', 'Mrs')
    
    data['Embarked'] = data['Embarked'].fillna('S')
    data['Fare'] = data['Fare'].fillna(data['Fare'].median())
    
    age_avg = data['Age'].mean()
    age_std = data['Age'].std()
    age_null_count = data['Age'].isnull().sum()
    age_null_random_list = np.random.randint(age_avg - age_std, age_avg + age_std, size=age_null_count)
    data.loc[data['Age'].isnull(), 'Age'] = age_null_random_list
    data['Age'] = data['Age'].astype(int)
    
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = 1
    data.loc[data['FamilySize'] > 1, 'IsAlone'] = 0
    
    data['Age*Class'] = data['Age'] * data['Pclass']
    data['Fare_per_Person'] = data['Fare'] / data['FamilySize']
    
    data['Cabin'] = data['Cabin'].fillna('Unknown')
    data['Deck'] = data['Cabin'].str.slice(0, 1)
    
    data['Age_Bin'] = pd.cut(data['Age'], bins=[0, 12, 20, 40, 60, 80, 100], labels=['Child', 'Teenager', 'Young Adult', 'Adult', 'Senior', 'Elderly'])
    data['Fare_Bin'] = pd.qcut(data['Fare'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])
    
    data.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)
    
    return data

train_data = preprocess_data(train_data)
test_data = preprocess_data(test_data)

# Define the features and target
features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'Age*Class', 'Fare_per_Person', 'Deck', 'Age_Bin', 'Fare_Bin']
target = 'Survived'

# Create the preprocessing pipeline
numeric_features = ['Age', 'Fare', 'FamilySize', 'Age*Class', 'Fare_per_Person']
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_features = ['Pclass', 'Sex', 'Embarked', 'Title', 'IsAlone', 'Deck', 'Age_Bin', 'Fare_Bin']
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Create the classifier pipeline
rf_clf = RandomForestClassifier(random_state=42)
gb_clf = GradientBoostingClassifier(random_state=42)
et_clf = ExtraTreesClassifier(random_state=42)

classifier = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', rf_clf)
])

# Define the hyperparameter grid for tuning
param_grid = {
    'classifier': [rf_clf, gb_clf, et_clf],
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 5, 10],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

# Perform grid search with cross-validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
grid_search = GridSearchCV(estimator=classifier, param_grid=param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid_search.fit(train_data[features], train_data[target])

# Get the best model and its performance
best_model = grid_search.best_estimator_
best_score = grid_search.best_score_
print(f"Best Model: {best_model}")
print(f"Best Score: {best_score:.4f}")

# Make predictions on the test data
test_predictions = best_model.predict(test_data[features])



Best Model: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare', 'FamilySize',
                                                   'Age*Class',
                                                   'Fare_per_Person']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Pclass', 'Sex', 'Embarked',
                                                   'Title', 'IsAlone', 'Deck',
                                                   'Age_Bin', 'Fare_Bin'])])),
                ('classifier',
    

In [2]:
# Save the predictions to a CSV file    
test_data = pd.read_csv('test.csv')
output = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Survived': test_predictions})

